# 🐄 Cattle Disease — Symptom Model Training
**4-class classifier: LSD · FMD · ECF · CBPP**

This notebook trains the symptom model used by the Cattle Disease mobile app.  
It runs end-to-end: data loading → feature engineering → hyperparameter search → evaluation → artifact export.

---
### Quick-start
1. **Runtime → Change runtime type → T4 GPU** (or CPU is fine — training is sklearn-based)  
2. Run **Cell 1** to install dependencies  
3. Run **Cell 2** to upload your `symptoms_merged.csv`  
4. Run the remaining cells in order  
5. Download `symptom_artifacts.zip` from the last cell

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
# scikit-learn 1.4+ is required for HistGradientBoostingClassifier class_weight
!pip install -q --upgrade scikit-learn==1.5.2 pandas numpy joblib matplotlib seaborn PyYAML

In [ ]:
# ── Cell 2: Upload data ───────────────────────────────────────────────────────
# Upload the merged symptom CSV (ml/data/processed/symptoms_merged.csv from your local project).
# Alternatively, mount Google Drive and set DRIVE_CSV_PATH below.

USE_DRIVE = False   # ← set True to load from Drive instead of file upload
DRIVE_CSV_PATH = '/content/drive/MyDrive/cattle_disease_ml/symptoms_merged.csv'

import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    CSV_PATH = DRIVE_CSV_PATH
    print(f'Using Drive CSV: {CSV_PATH}')
else:
    from google.colab import files
    print('Please upload symptoms_merged.csv ...')
    uploaded = files.upload()
    fname = next(iter(uploaded))
    os.rename(fname, '/content/symptoms_merged.csv')
    CSV_PATH = '/content/symptoms_merged.csv'
    print(f'Uploaded: {CSV_PATH}')

In [ ]:
# ── Cell 3: Imports & config ──────────────────────────────────────────────────
import math, json, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import joblib
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from sklearn.inspection import permutation_importance

warnings.filterwarnings('ignore', category=UserWarning)

# ── Training config (mirrors ml/configs/config.yaml) ─────────────────────────
CFG = {
    'seed': 42,
    'symptom_labels': ['CBPP', 'ECF', 'FMD', 'LSD'],   # Normal excluded — image-only
    'model_candidates': ['hist_gradient_boosting', 'random_forest', 'extra_trees'],
    'search_n_iter': 40,
    'cv_folds': 5,
    'n_estimators': 300,
    'max_depth': 12,
    'min_samples_leaf': 6,
    'target_macro_f1': 0.90,
    'max_overfit_gap': 0.06,
    'train_frac': 0.70,
    'val_frac':   0.15,
    'test_frac':  0.15,
}

ARTIFACTS_DIR = Path('/content/artifacts')
ARTIFACTS_DIR.mkdir(exist_ok=True)
print('Config ready. Artifacts will be saved to:', ARTIFACTS_DIR)

In [ ]:
# ── Cell 4: Feature engineering (mirrors ml/src/data/feature_engineering.py) ─

def _col(df, name, default=0.0):
    if name in df.columns:
        return df[name].fillna(default).astype(float)
    return pd.Series(default, index=df.index, dtype=float)

def add_engineered_features(df):
    df = df.copy()

    # Disease core scores
    df['fe_lsd_core']  = (_col(df,'skin_nodules') + _col(df,'painless_lumps') + _col(df,'enlarged_lymph_nodes')) / 3.0
    df['fe_fmd_core']  = (_col(df,'mouth_blisters') + _col(df,'tongue_sores') + _col(df,'foot_lesions') + _col(df,'drooling')) / 4.0
    df['fe_ecf_core']  = (_col(df,'swollen_lymph_nodes') + _col(df,'eye_discharge') + _col(df,'difficulty_breathing') + _col(df,'loss_of_appetite')) / 4.0
    df['fe_cbpp_core'] = (_col(df,'coughing') + _col(df,'chest_pain_signs') + _col(df,'rapid_shallow_breathing') + _col(df,'difficulty_breathing')) / 4.0

    # Pairwise disambiguation scores
    df['fe_lsd_vs_fmd']   = _col(df,'skin_nodules') + _col(df,'painless_lumps') - _col(df,'mouth_blisters') - _col(df,'tongue_sores')
    df['fe_ecf_vs_cbpp']  = _col(df,'swollen_lymph_nodes') + _col(df,'eye_discharge') - _col(df,'coughing') - _col(df,'chest_pain_signs') - _col(df,'rapid_shallow_breathing')

    # Aggregate symptom burden
    _BINARY = ['chest_pain_signs','coughing','depression','diarrhoea','difficulty_breathing',
               'drooling','enlarged_lymph_nodes','eye_discharge','fever','foot_lesions',
               'lameness','loss_of_appetite','mouth_blisters','nasal_discharge','painless_lumps',
               'rapid_shallow_breathing','skin_nodules','swollen_lymph_nodes','tongue_sores']
    avail = [c for c in _BINARY if c in df.columns]
    df['fe_total_symptoms'] = df[avail].fillna(0).astype(float).sum(axis=1)
    df['fe_respiratory'] = _col(df,'coughing') + _col(df,'difficulty_breathing') + _col(df,'rapid_shallow_breathing') + _col(df,'chest_pain_signs') + _col(df,'nasal_discharge')
    df['fe_skin']        = _col(df,'skin_nodules') + _col(df,'painless_lumps') + _col(df,'enlarged_lymph_nodes')
    df['fe_oral_foot']   = _col(df,'mouth_blisters') + _col(df,'tongue_sores') + _col(df,'foot_lesions') + _col(df,'drooling')
    df['fe_lymph']       = _col(df,'swollen_lymph_nodes') + _col(df,'enlarged_lymph_nodes')

    # Pathognomonic interactions
    fever = _col(df,'fever')
    df['fe_fever_x_skin']  = fever * _col(df,'skin_nodules')
    df['fe_fever_x_lymph'] = fever * _col(df,'swollen_lymph_nodes')
    df['fe_fever_x_cough'] = fever * _col(df,'coughing')
    df['fe_fever_x_oral']  = fever * _col(df,'mouth_blisters')
    df['fe_tick_x_lymph']  = _col(df,'tick_exposure_recent') * _col(df,'swollen_lymph_nodes')
    df['fe_cough_x_chest'] = _col(df,'coughing') * _col(df,'chest_pain_signs')
    df['fe_oral_x_foot']   = _col(df,'mouth_blisters') * _col(df,'foot_lesions')

    # Temporal
    if 'days_since_onset' in df.columns:
        onset = df['days_since_onset'].fillna(0).clip(lower=0).astype(float)
        df['fe_acute_onset'] = (onset <= 5).astype(float)
        df['fe_log_onset']   = onset.apply(math.log1p)
    else:
        df['fe_acute_onset'] = 0.0
        df['fe_log_onset']   = 0.0

    df['fe_severe_burden'] = _col(df,'case_severity_severe') * df['fe_total_symptoms'] if 'case_severity_severe' in df.columns else 0.0
    return df

print('Feature engineering function loaded (21 fe_* features).')

In [ ]:
# ── Cell 5: Load & explore data ───────────────────────────────────────────────
df_raw = pd.read_csv(CSV_PATH)
print(f'Raw shape: {df_raw.shape}')
print(f'Columns: {list(df_raw.columns)}')
print()

# Drop Normal — symptom model is 4-class only
if 'Normal' in df_raw['Disease'].unique():
    n_dropped = (df_raw['Disease'] == 'Normal').sum()
    df_raw = df_raw[df_raw['Disease'] != 'Normal'].copy()
    print(f'Dropped {n_dropped} Normal rows — Normal is handled by the image model only.')

print(f'\nClass distribution after filtering:')
print(df_raw['Disease'].value_counts().to_string())
print(f'\nTotal rows: {len(df_raw)}  |  Unique rows: {len(df_raw.drop_duplicates())}')

# Quick class balance bar chart
fig, ax = plt.subplots(figsize=(7, 3))
counts = df_raw['Disease'].value_counts()
colors = {'CBPP': '#2E7BA0', 'ECF': '#7B5EA7', 'FMD': '#D94F3D', 'LSD': '#E6A817'}
bars = ax.barh(counts.index, counts.values, color=[colors.get(d, '#888') for d in counts.index], height=0.5)
for bar, v in zip(bars, counts.values):
    ax.text(bar.get_width() + 30, bar.get_y() + bar.get_height()/2, f'{v:,}', va='center', fontsize=10)
ax.set_xlabel('Samples')
ax.set_title('Class distribution (symptom model)')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'class_distribution.png', dpi=120)
plt.show()

In [ ]:
# ── Cell 6: Feature engineering + split ──────────────────────────────────────
RESERVED_COLS = {'Disease'}

df_raw['__source'] = 'real'
RESERVED_COLS.add('__source')

# Apply feature engineering to the full dataset BEFORE splitting
print('Applying feature engineering...')
df_eng = add_engineered_features(df_raw)

feature_cols = [c for c in df_eng.columns if c not in RESERVED_COLS]
base_count = len([c for c in feature_cols if not c.startswith('fe_')])
fe_count   = len([c for c in feature_cols if c.startswith('fe_')])
print(f'Features: {base_count} base + {fe_count} engineered = {len(feature_cols)} total')

# 70 / 15 / 15 stratified split
df_train, df_hold = train_test_split(
    df_eng, test_size=1 - CFG['train_frac'],
    stratify=df_eng['Disease'], random_state=CFG['seed']
)
df_val, df_test = train_test_split(
    df_hold, test_size=0.5,
    stratify=df_hold['Disease'], random_state=CFG['seed']
)

print(f'\nSplit sizes  → train: {len(df_train):,}  val: {len(df_val):,}  test: {len(df_test):,}')
print('Train class counts:', df_train['Disease'].value_counts().to_dict())

X_train = df_train[feature_cols].astype(float).values
y_train = df_train['Disease'].astype(str).values
X_val   = df_val[feature_cols].astype(float).values
y_val   = df_val['Disease'].astype(str).values
X_test  = df_test[feature_cols].astype(float).values
y_test  = df_test['Disease'].astype(str).values

In [ ]:
# ── Cell 7: Model candidate definitions ──────────────────────────────────────

def macro_f1(y_true, y_pred):
    labels = sorted(set(y_true.tolist()) | set(y_pred.tolist()))
    return float(f1_score(y_true, y_pred, labels=labels, average='macro', zero_division=0))

def build_candidate(name, seed=42):
    if name == 'hist_gradient_boosting':
        model = HistGradientBoostingClassifier(
            max_iter=300, learning_rate=0.10, max_leaf_nodes=31,
            min_samples_leaf=20, l2_regularization=0.05,
            class_weight='balanced', random_state=seed, early_stopping=False,
        )
        params = {
            'learning_rate':     [0.04, 0.06, 0.08, 0.10, 0.12, 0.15],
            'max_iter':          [200, 300, 400, 500],
            'max_leaf_nodes':    [20, 31, 47, 63],
            'min_samples_leaf':  [10, 15, 20, 30, 40],
            'l2_regularization': [0.0, 0.01, 0.05, 0.10, 0.50],
        }
    elif name == 'extra_trees':
        model = ExtraTreesClassifier(
            n_estimators=300, max_depth=12, min_samples_leaf=6,
            bootstrap=True, random_state=seed,
            class_weight='balanced', n_jobs=-1,
        )
        params = {
            'n_estimators':      [200, 300, 400],
            'max_depth':         [8, 10, 12, 15],
            'min_samples_leaf':  [4, 6, 8, 10],
            'min_samples_split': [8, 12, 16, 20],
            'max_features':      ['sqrt', 'log2', 0.40, 0.50],
            'bootstrap':         [True],
        }
    else:  # random_forest
        model = RandomForestClassifier(
            n_estimators=300, max_depth=12, min_samples_leaf=6,
            bootstrap=True, random_state=seed,
            class_weight='balanced_subsample', n_jobs=-1,
        )
        params = {
            'n_estimators':      [200, 300, 400],
            'max_depth':         [8, 10, 12, 15],
            'min_samples_leaf':  [4, 6, 8, 10],
            'min_samples_split': [8, 12, 16, 20],
            'max_features':      ['sqrt', 'log2', 0.40, 0.50],
            'bootstrap':         [True],
        }
    return model, params

print('Model candidates ready:', CFG['model_candidates'])

In [ ]:
# ── Cell 8: Hyperparameter search (all candidates) ────────────────────────────
# This is the main training cell. Expect ~10-25 min for all 3 candidates.
# You can reduce search_n_iter in CFG (e.g. to 20) for a faster run.

cv = StratifiedKFold(n_splits=CFG['cv_folds'], shuffle=True, random_state=CFG['seed'])

all_results = []
best_model  = None
best_val_f1 = -1.0

for name in CFG['model_candidates']:
    print(f'\n{'='*60}')
    print(f'  Training: {name}')
    print(f'{'='*60}')
    t0 = time.time()

    estimator, param_space = build_candidate(name, seed=CFG['seed'])
    search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=param_space,
        n_iter=CFG['search_n_iter'],
        scoring='f1_macro',
        n_jobs=-1,
        cv=cv,
        random_state=CFG['seed'],
        refit=True,
        verbose=1,
        error_score='raise',
    )
    search.fit(X_train, y_train)
    fitted = search.best_estimator_

    train_pred = fitted.predict(X_train)
    val_pred   = fitted.predict(X_val)
    train_f1   = macro_f1(y_train, train_pred)
    val_f1     = macro_f1(y_val, val_pred)
    gap        = train_f1 - val_f1
    elapsed    = time.time() - t0

    print(f'  cv_f1={search.best_score_:.4f}  train_f1={train_f1:.4f}  '
          f'val_f1={val_f1:.4f}  gap={gap:.4f}  ({elapsed:.0f}s)')
    print(f'  Best params: {search.best_params_}')

    all_results.append({
        'name':           name,
        'cv_macro_f1':    round(search.best_score_, 6),
        'train_macro_f1': round(train_f1, 6),
        'val_macro_f1':   round(val_f1, 6),
        'overfit_gap':    round(gap, 6),
        'best_params':    search.best_params_,
        'model':          fitted,
    })

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model  = fitted
        best_name   = name

print(f'\n{'*'*60}')
print(f'  Winner: {best_name}  val_f1={best_val_f1:.4f}')
print(f'{'*'*60}')

In [ ]:
# ── Cell 9: Candidate comparison bar chart ────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
x   = np.arange(len(all_results))
w   = 0.25
ax.bar(x - w, [r['cv_macro_f1']    for r in all_results], w, label='CV macro-F1',    color='#4C9A6E')
ax.bar(x,     [r['train_macro_f1'] for r in all_results], w, label='Train macro-F1', color='#E6A817')
ax.bar(x + w, [r['val_macro_f1']   for r in all_results], w, label='Val macro-F1',   color='#2E7BA0')
ax.axhline(CFG['target_macro_f1'], color='red', ls='--', lw=1.2, label=f'Target {CFG["target_macro_f1"]}')
ax.set_xticks(x)
ax.set_xticklabels([r['name'].replace('_', '\n') for r in all_results])
ax.set_ylim(0.75, 1.02)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
ax.set_ylabel('Macro F1')
ax.set_title('Model candidate comparison')
ax.legend(loc='lower right')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'candidate_comparison.png', dpi=120)
plt.show()

In [ ]:
# ── Cell 10: Test-set evaluation ─────────────────────────────────────────────
y_pred_test = best_model.predict(X_test)
test_f1     = macro_f1(y_test, y_pred_test)
gap_train   = [r for r in all_results if r['name'] == best_name][0]['overfit_gap']
val_f1_best = [r for r in all_results if r['name'] == best_name][0]['val_macro_f1']

print('=' * 60)
print(f'  BEST MODEL : {best_name}')
print(f'  Train F1   : {[r for r in all_results if r["name"]==best_name][0]["train_macro_f1"]:.4f}')
print(f'  Val   F1   : {val_f1_best:.4f}')
print(f'  Test  F1   : {test_f1:.4f}   ← hold-out')
print(f'  Overfit gap: {gap_train:.4f}  (target ≤ {CFG["max_overfit_gap"]})')
print('=' * 60)

# Quality gate messages
warns = []
if gap_train > CFG['max_overfit_gap']:
    warns.append(f'[WARN] Overfit gap {gap_train:.4f} exceeds max {CFG["max_overfit_gap"]}')
if test_f1 < CFG['target_macro_f1']:
    warns.append(f'[WARN] Test F1 {test_f1:.4f} below target {CFG["target_macro_f1"]}')
if not warns:
    print('\n✅ All quality gates passed!')
else:
    for w in warns:
        print(w)

# Per-class report
test_report = classification_report(y_test, y_pred_test, zero_division=0, output_dict=True)
print()
print(classification_report(y_test, y_pred_test, zero_division=0))

In [ ]:
# ── Cell 11: Confusion matrix ─────────────────────────────────────────────────
labels_ordered = sorted(CFG['symptom_labels'])
cm = confusion_matrix(y_test, y_pred_test, labels=labels_ordered)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=labels_ordered, yticklabels=labels_ordered,
            linewidths=0.5, ax=ax, cbar_kws={'label': '%'})
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True', fontsize=12)
ax.set_title(f'Confusion matrix — {best_name}\nTest macro-F1 = {test_f1:.4f}', fontsize=13)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'confusion_matrix_symptom.png', dpi=150)
plt.show()

In [ ]:
# ── Cell 12: Per-class F1 bar chart ──────────────────────────────────────────
DISEASE_COLORS = {'CBPP': '#2E7BA0', 'ECF': '#7B5EA7', 'FMD': '#D94F3D', 'LSD': '#E6A817'}
per_class_f1 = {d: test_report[d]['f1-score'] for d in labels_ordered if d in test_report}

fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.bar(list(per_class_f1.keys()), list(per_class_f1.values()),
              color=[DISEASE_COLORS.get(d,'#888') for d in per_class_f1],
              width=0.5, edgecolor='white', linewidth=1.2)
for bar, (d, v) in zip(bars, per_class_f1.items()):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.005, f'{v:.3f}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.axhline(CFG['target_macro_f1'], color='red', ls='--', lw=1.2, label=f'Target {CFG["target_macro_f1"]}')
ax.set_ylim(0.70, 1.02)
ax.set_ylabel('F1-score')
ax.set_title('Per-class F1 (test set)')
ax.legend()
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'per_class_f1.png', dpi=120)
plt.show()

In [ ]:
# ── Cell 13: Feature importance ───────────────────────────────────────────────
if hasattr(best_model, 'feature_importances_'):
    importances = np.asarray(best_model.feature_importances_, dtype=float)
else:
    print('Computing permutation importance (may take a few minutes)...')
    perm = permutation_importance(best_model, X_val, y_val,
                                  n_repeats=5, random_state=CFG['seed'],
                                  scoring='f1_macro', n_jobs=-1)
    importances = np.asarray(perm.importances_mean, dtype=float)

# Top 20, excluding fe_* engineered features
raw_features = [(feature_cols[i], importances[i])
                for i in range(len(feature_cols))
                if not feature_cols[i].startswith('fe_')]
raw_features.sort(key=lambda x: x[1], reverse=True)
top20 = raw_features[:20]

fig, ax = plt.subplots(figsize=(9, 6))
names, imps = zip(*top20)
y_pos = np.arange(len(names))
ax.barh(y_pos, imps, color='#4C9A6E', height=0.6)
ax.set_yticks(y_pos)
ax.set_yticklabels(names, fontsize=10)
ax.invert_yaxis()
ax.set_xlabel('Feature importance')
ax.set_title(f'Top 20 raw symptom importances — {best_name}')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'feature_importance.png', dpi=120)
plt.show()

top_features_list = [
    {'feature': feature_cols[i], 'importance': round(float(importances[i]), 6)}
    for i in np.argsort(importances)[::-1][:20]
]

In [ ]:
# ── Cell 14: Save model artifacts ────────────────────────────────────────────
import sklearn
# symptom_model.pkl
model_path = ARTIFACTS_DIR / 'symptom_model.pkl'
joblib.dump(best_model, model_path, compress=3)
model_size_mb = model_path.stat().st_size / 1e6
print(f'[save] symptom_model.pkl  ({model_size_mb:.1f} MB)')

# symptom_features.json
features_path = ARTIFACTS_DIR / 'symptom_features.json'
features_path.write_text(json.dumps(feature_cols, indent=2))
print(f'[save] symptom_features.json  ({len(feature_cols)} features)')

# label_map.json
labels_trained = sorted(df_eng['Disease'].unique().tolist())
label_map = {
    'final_labels':           ['Normal', 'LSD', 'FMD', 'ECF', 'CBPP'],
    'symptom_train_labels':   labels_trained,
    'image_labels':           ['Normal', 'LSD', 'FMD'],
    'symptom_training_mode':  'real_only',
}
label_map_path = ARTIFACTS_DIR / 'label_map.json'
label_map_path.write_text(json.dumps(label_map, indent=2))
print(f'[save] label_map.json')

# symptom_model_metadata.json
best_result = next(r for r in all_results if r['name'] == best_name)
metadata = {
    'training_mode':         'real_only',
    'real_unique_rows':      int(len(df_raw.drop_duplicates())),
    'real_rows':             int(len(df_raw)),
    'synthetic_rows_added':  0,
    'labels_trained':        labels_trained,
    'selected_model':        type(best_model).__name__,
    'selected_candidate':    best_name,
    'train_macro_f1':        best_result['train_macro_f1'],
    'val_macro_f1':          best_result['val_macro_f1'],
    'test_macro_f1':         round(test_f1, 6),
    'overfit_gap_train_val': best_result['overfit_gap'],
    'target_macro_f1':       CFG['target_macro_f1'],
    'max_overfit_gap':       CFG['max_overfit_gap'],
    'candidate_results':     [{k: v for k, v in r.items() if k != 'model'} for r in all_results],
    'top_features':          top_features_list,
    'test_classification_report': test_report,
    'sklearn_version':       sklearn.__version__,
    'warnings':              warns,
}
meta_path = ARTIFACTS_DIR / 'symptom_model_metadata.json'
meta_path.write_text(json.dumps(metadata, indent=2))
print(f'[save] symptom_model_metadata.json')

# Test predictions CSV
pd.DataFrame({'true': y_test, 'pred': y_pred_test}).to_csv(
    ARTIFACTS_DIR / 'symptom_test_predictions.csv', index=False
)
print(f'[save] symptom_test_predictions.csv')

print(f'\nAll artifacts saved to: {ARTIFACTS_DIR}')

In [ ]:
# ── Cell 15: Print final summary ──────────────────────────────────────────────
print('\n' + '='*65)
print('  SYMPTOM MODEL TRAINING SUMMARY')
print('='*65)
print(f'  Selected model    : {type(best_model).__name__}  ({best_name})')
print(f'  Model size        : {model_size_mb:.1f} MB')
print(f'  Classes           : {labels_trained}')
print(f'  Features (total)  : {len(feature_cols)}')
print()
print(f'  Metrics:')
best_r = next(r for r in all_results if r['name'] == best_name)
print(f'    Train macro-F1  : {best_r["train_macro_f1"]:.4f}')
print(f'    Val   macro-F1  : {best_r["val_macro_f1"]:.4f}')
print(f'    Test  macro-F1  : {test_f1:.4f}')
print(f'    Overfit gap     : {best_r["overfit_gap"]:.4f}  (limit {CFG["max_overfit_gap"]})')
print()
print(f'  Per-class F1 (test):')
for d in labels_ordered:
    if d in test_report:
        r = test_report[d]
        print(f'    {d:8s}  P={r["precision"]:.3f}  R={r["recall"]:.3f}  F1={r["f1-score"]:.3f}  n={int(r["support"])}')
print('='*65)
if warns:
    for w in warns:
        print(w)
else:
    print('  ✅ All quality gates passed — model is ready for deployment')
print('='*65)

In [ ]:
# ── Cell 16: Download all artifacts as zip ────────────────────────────────────
import shutil
from google.colab import files

zip_path = '/content/symptom_artifacts'
shutil.make_archive(zip_path, 'zip', ARTIFACTS_DIR)
print(f'Created: {zip_path}.zip')
print('After downloading, extract and copy into cattle_disease_ml/ml/artifacts/')
print('Required files:')
for f in ARTIFACTS_DIR.iterdir():
    sz = f.stat().st_size
    print(f'  {f.name:45s}  {sz/1e6:6.2f} MB' if sz > 1e5 else f'  {f.name}')
print()
files.download(f'{zip_path}.zip')

## After downloading

1. Extract `symptom_artifacts.zip`
2. Copy these files into your local `cattle_disease_ml/ml/artifacts/`:
   - `symptom_model.pkl` ← trained model
   - `symptom_features.json` ← feature list for inference
   - `label_map.json` ← class mapping
   - `symptom_model_metadata.json` ← training metrics
3. Copy PNG charts into `ml/artifacts/reports/` (optional)
4. Rebuild and redeploy the `ml` Docker service:
   ```bash
   docker compose build ml && docker compose up -d ml
   ```

**Scikit-learn version note:** The model was saved with scikit-learn `x.y.z`.  
Make sure your deployment environment uses the **same major.minor version** to avoid deserialization issues.  
Check the saved `symptom_model_metadata.json` → `sklearn_version` field.